# 🧠 Transfer Learning in Deep Learning (MobileNetV2 & ResNet50)

In this notebook we will apply **Transfer Learning** to classify the CIFAR-10 dataset. We build on the concepts covered in `EXTRA Transfer Learning II.ipynb`.

## 📌 What is Transfer Learning?
Transfer Learning consists of taking a model that has already been pre-trained on a massive dataset and adapting it for our own problem.

The process is divided into two main phases:

1. **Feature Extraction:**
   - We import the pre-trained base model without the final classification layer (`include_top=False`).
   - We **freeze** the base model weights (`trainable = False`) so they are not modified.
   - We add our own classification "head" (Dense layers) at the end.
   - We train **only** our new classification head. The base model acts solely as a generic "feature extractor".

2. **Fine-Tuning:**
   - Once our classification head has learned initial patterns, we **unfreeze** some or all of the top layers of the base model.
   - We retrain using a **very small Learning Rate**.
   - This allows the model to subtly adapt its prior knowledge (weights) to the specific characteristics of our images.


In [ ]:
# 1. Setup and imports for training (Google Colab)
import os
import json
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from keras.datasets import cifar10
from keras.utils import to_categorical
from keras.layers import Dense, GlobalAveragePooling2D, Input, Dropout
from keras.models import Sequential, Model
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam

# Configuration
BATCH_SIZE = 64
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer','dog', 'frog', 'horse', 'ship', 'truck']


In [ ]:
# 2. Load and prepare data
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = cifar10.load_data()

# One-hot encode labels
y_train = to_categorical(y_train_raw, 10)
y_test = to_categorical(y_test_raw, 10)

# Split training set for validation
from sklearn.model_selection import train_test_split
x_train_split, x_val, y_train_split, y_val = train_test_split(
    x_train_raw, y_train, test_size=0.2, random_state=42
)

# Resize for Transfer Learning models: 32x32 -> 96x96 (bilinear interpolation)
# Pre-trained models need larger images to extract features correctly.
# We pre-resize once here to avoid doing it on every batch during training.
x_train_split_96 = tf.image.resize(x_train_split, [96, 96]).numpy()
x_val_96 = tf.image.resize(x_val, [96, 96]).numpy()

print(f"X_train_split shape (original): {x_train_split.shape}")
print(f"X_train_split_96 shape (resized): {x_train_split_96.shape}")
print(f"X_val shape: {x_val.shape}")
print(f"X_val_96 shape (resized): {x_val_96.shape}")
print("Data ready for training.")


---
## 🚀 Model 1: MobileNetV2

In [ ]:
# MobileNetV2 - PHASE 1: Feature Extraction
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input

# Preprocess the RESIZED images using the model-specific function
x_train_mb = preprocess_input(x_train_split_96.astype('float32'))
x_val_mb = preprocess_input(x_val_96.astype('float32'))

# Load base model with 96x96 input and FREEZE it
base_model_mb = MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights='imagenet')
base_model_mb.trainable = False

# Build our architecture
model_mb = Sequential([
    base_model_mb,
    GlobalAveragePooling2D(),
    Dropout(0.2), # To prevent overfitting
    Dense(10, activation='softmax')
])

model_mb.compile(optimizer=Adam(learning_rate=0.001),
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

print("Starting Phase 1: Feature Extraction...")
history_mb_fe = model_mb.fit(x_train_mb, y_train_split, epochs=5, validation_data=(x_val_mb, y_val), batch_size=BATCH_SIZE)


In [ ]:
# MobileNetV2 - PHASE 2: Fine-Tuning

# Unfreeze the base model
base_model_mb.trainable = True

# Recompile with a MUCH SMALLER Learning Rate (1e-5 or 1e-4) to avoid destroying weights
model_mb.compile(optimizer=Adam(learning_rate=1e-5),
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

print("Starting Phase 2: Fine-Tuning...")
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_mb_ft = model_mb.fit(x_train_mb, y_train_split, epochs=10, 
                             validation_data=(x_val_mb, y_val), 
                             batch_size=BATCH_SIZE, callbacks=[early_stop])

# Combine training histories to plot everything together
acc_mbV2 = history_mb_fe.history['accuracy'] + history_mb_ft.history['accuracy']
val_acc_mbV2 = history_mb_fe.history['val_accuracy'] + history_mb_ft.history['val_accuracy']
loss_mbV2 = history_mb_fe.history['loss'] + history_mb_ft.history['loss']
val_loss_mbV2 = history_mb_fe.history['val_loss'] + history_mb_ft.history['val_loss']

# Save training plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(acc_mbV2, label='Train Accuracy')
axes[0].plot(val_acc_mbV2, label='Validation Accuracy')
axes[0].axvline(len(history_mb_fe.history['accuracy'])-1, color='r', linestyle='--', label='Start Fine Tuning')
axes[0].set_title('MobileNetV2 Accuracy')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(loss_mbV2, label='Train Loss')
axes[1].plot(val_loss_mbV2, label='Validation Loss')
axes[1].axvline(len(history_mb_fe.history['loss'])-1, color='r', linestyle='--', label='Start Fine Tuning')
axes[1].set_title('MobileNetV2 Loss')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].legend()

# Save plot locally (On Colab it will be in the environment files)
plt.tight_layout()
plt.savefig('mobilenetv2_history.png')
print("✅ History saved as mobilenetv2_history.png")
plt.show()

# Save the trained model
model_mb.save('mobilenetv2_tl.keras')
print("✅ MobileNetV2 model saved as mobilenetv2_tl.keras")


---
## 🚀 Model 2: ResNet50

In [ ]:
# ResNet50 - PHASE 1: Feature Extraction
from keras.applications import ResNet50
from keras.applications.resnet50 import preprocess_input as preprocess_input_rn

# Preprocess the RESIZED images using the ResNet50-specific function
x_train_rn = preprocess_input_rn(x_train_split_96.astype('float32'))
x_val_rn = preprocess_input_rn(x_val_96.astype('float32'))

# Load base model with 96x96 input and FREEZE it
base_model_rn = ResNet50(input_shape=(96, 96, 3), include_top=False, weights='imagenet')
base_model_rn.trainable = False

# Build our architecture
model_rn = Sequential([
    base_model_rn,
    GlobalAveragePooling2D(),
    Dropout(0.2), # Added to prevent overfitting
    Dense(10, activation='softmax')
])

model_rn.compile(optimizer=Adam(learning_rate=0.001),
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

print("Starting Phase 1: Feature Extraction for ResNet50...")
history_rn_fe = model_rn.fit(x_train_rn, y_train_split, epochs=5, validation_data=(x_val_rn, y_val), batch_size=BATCH_SIZE)


In [ ]:
# ResNet50 - PHASE 2: Fine-Tuning

# Unfreeze the base model
base_model_rn.trainable = True

# Recompile with a MUCH SMALLER Learning Rate
model_rn.compile(optimizer=Adam(learning_rate=1e-5),
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

print("Starting Phase 2: Fine-Tuning for ResNet50...")
early_stop_rn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_rn_ft = model_rn.fit(x_train_rn, y_train_split, epochs=10, 
                             validation_data=(x_val_rn, y_val), 
                             batch_size=BATCH_SIZE, callbacks=[early_stop_rn])

# Combine training histories
acc_rn50 = history_rn_fe.history['accuracy'] + history_rn_ft.history['accuracy']
val_acc_rn50 = history_rn_fe.history['val_accuracy'] + history_rn_ft.history['val_accuracy']
loss_rn50 = history_rn_fe.history['loss'] + history_rn_ft.history['loss']
val_loss_rn50 = history_rn_fe.history['val_loss'] + history_rn_ft.history['val_loss']

# Save training plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(acc_rn50, label='Train Accuracy')
axes[0].plot(val_acc_rn50, label='Validation Accuracy')
axes[0].axvline(len(history_rn_fe.history['accuracy'])-1, color='r', linestyle='--', label='Start Fine Tuning')
axes[0].set_title('ResNet50 Accuracy')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(loss_rn50, label='Train Loss')
axes[1].plot(val_loss_rn50, label='Validation Loss')
axes[1].axvline(len(history_rn_fe.history['loss'])-1, color='r', linestyle='--', label='Start Fine Tuning')
axes[1].set_title('ResNet50 Loss')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('resnet50_history.png')
print("✅ History saved as resnet50_history.png")
plt.show()

# Save the trained model
model_rn.save('resnet50_tl.keras')
print("✅ ResNet50 model saved as resnet50_tl.keras")


---
## 📊 Local Evaluation and Model Comparison

**IMPORTANT NOTE:** Run the cells below **after** training on Google Colab and downloading the models `mobilenetv2_tl.keras` and `resnet50_tl.keras` to your local machine, placing them inside the `models` folder (`/Users/sebastianlopez/Desktop/it-studies/ironhack/week_7/day_2/models`).

In this section we will:
1. Load the Custom CNN and compute its metrics on the Test dataset.
2. Load MobileNetV2 and ResNet50, preprocessing Test images according to each model's requirements.
3. Evaluate and compute `accuracy`, `precision`, `recall` and `f1-score`.
4. Export the new models' metrics as `.json`.
5. Plot and save comparisons (`model_comparison_mobilenetv2_cnn.png` and `model_comparison_resnet50_cnn.png`).


In [ ]:
# 3. Local Evaluation and JSON Export
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

from tensorflow import keras
from keras.datasets import cifar10
from keras.utils import to_categorical
from keras.applications.mobilenet_v2 import preprocess_input as mb_prep
from keras.applications.resnet50 import preprocess_input as rn_prep

# Load test data
(_, _), (x_test_raw, y_test_raw) = cifar10.load_data()
y_test = to_categorical(y_test_raw, 10)

# Preprocessing for the test set (same as during training)
x_test_custom = x_test_raw.astype('float32') / 255.0

# Resize test images to 96x96 for Transfer Learning models
# (same resize applied to training data)
x_test_96 = tf.image.resize(x_test_raw, [96, 96]).numpy()
x_test_mb = mb_prep(x_test_96.astype('float32'))
x_test_rn = rn_prep(x_test_96.astype('float32'))

# Paths - Make sure the files exist locally
MODEL_DIR = '/Users/sebastianlopez/Desktop/it-studies/ironhack/week_7/day_2/models'
OUTPUT_DIR = '/Users/sebastianlopez/Desktop/it-studies/ironhack/week_7/day_2/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def evaluate_and_save(model_path, x_test_prep, model_name, file_suffix):
    if not os.path.exists(model_path):
        print(f"❌ Model not found at: {model_path}")
        return None
    
    print(f"\nLoading and evaluating {model_name}...")
    model = keras.models.load_model(model_path)
    loss, accuracy = model.evaluate(x_test_prep, y_test, verbose=0)
    
    y_pred_proba = model.predict(x_test_prep, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = np.argmax(y_test, axis=1)
    
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    print(f"  Test Loss:     {loss:.4f}")
    print(f"  Test Accuracy: {accuracy:.4f}")
    print(f"  Precision:     {prec:.4f}")
    print(f"  Recall:        {rec:.4f}")
    print(f"  F1-Score:      {f1:.4f}")
    
    metrics = {
        "model": model_name,
        "loss": float(loss),
        "accuracy": float(accuracy),
        "precision": float(prec),
        "recall": float(rec),
        "f1_score": float(f1)
    }
    
    # Export JSON
    json_path = os.path.join(OUTPUT_DIR, f"{file_suffix}_metrics.json")
    with open(json_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"✅ Metrics exported to {json_path}")
    
    return metrics

# Load metrics for all 3 models
custom_cnn_path = os.path.join(MODEL_DIR, 'custom_cnn.keras')
mb_path = os.path.join(MODEL_DIR, 'mobilenetv2_tl.keras')
rn_path = os.path.join(MODEL_DIR, 'resnet50_tl.keras')

metrics_custom = evaluate_and_save(custom_cnn_path, x_test_custom, 'Custom CNN', 'custom_cnn')
metrics_mb = evaluate_and_save(mb_path, x_test_mb, 'MobileNetV2', 'mobilenetv2_tl')
metrics_rn = evaluate_and_save(rn_path, x_test_rn, 'ResNet50', 'resnet50_tl')


In [ ]:
# 4. Comparison Chart: Custom CNN vs MobileNetV2
def plot_comparison(m1, m2, save_filename, title):
    if m1 is None or m2 is None:
        print(f"Missing data to plot {title}")
        return
    
    labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    values_m1 = [m1['accuracy'], m1['precision'], m1['recall'], m1['f1_score']]
    values_m2 = [m2['accuracy'], m2['precision'], m2['recall'], m2['f1_score']]
    
    x = np.arange(len(labels))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(8, 5))
    rects1 = ax.bar(x - width/2, values_m1, width, label=m1['model'], color='skyblue')
    rects2 = ax.bar(x + width/2, values_m2, width, label=m2['model'], color='salmon')
    
    ax.set_ylabel('Scores')
    ax.set_title(title, fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend(loc='lower right')
    ax.set_ylim(0, 1.1)
    
    # Add numeric values on top of bars
    for rects in [rects1, rects2]:
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.3f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), 
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)
            
    # Add box with Loss info
    textstr = f"Test Loss {m1['model']}: {m1['loss']:.4f}\nTest Loss {m2['model']}: {m2['loss']:.4f}"
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(1.05, 0.5, textstr, transform=ax.transAxes, fontsize=10, verticalalignment='center', bbox=props)
            
    # Save locally
    save_path = os.path.join(OUTPUT_DIR, save_filename)
    plt.savefig(save_path, bbox_inches='tight')
    print(f"✅ Comparison saved at: {save_path}")
    plt.show()

# Run and plot the first comparison
plot_comparison(metrics_custom, metrics_mb, 'model_comparison_mobilenetv2_cnn.png', 'Comparison: Custom CNN vs MobileNetV2')


In [ ]:
# 5. Comparison Chart: Custom CNN vs ResNet50
# Run and plot the second comparison
plot_comparison(metrics_custom, metrics_rn, 'model_comparison_resnet50_cnn.png', 'Comparison: Custom CNN vs ResNet50')
